# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/124pritivarma6001-commits/flyrank_internship_ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

This section defines the feature window used for the leakage audit.  
The March 2026 daily performance data is loaded at the client-content level using measurable SEO and engagement features.

The selected features include Google Search Console impressions, clicks, average position, and other available performance signals. The feature dataset is kept separate from the future outcome label so that the prediction setup can be evaluated for potential leakage.

In [1]:
%pip -q install duckdb huggingface_hub

In [2]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

Paste your Hugging Face READ token (hf_...): ··········


In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [5]:
# ML-05 — Feature Leakage Check
# Section 1: Build the feature vector

daily_table = TABLES["fact_daily"]

feature_data = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_sessions,
    ga4_users,
    ga4_engaged_sessions,
    ga4_total_engagement_sec,
    sessions_organic,
    sessions_direct,
    sessions_referral,
    sessions_social,
    sessions_paid,
    sessions_ai,
    ai_chatgpt,
    ai_perplexity,
    ai_gemini,
    ai_copilot,
    ai_claude,
    ai_meta,
    ai_other,
    scroll_events
FROM {daily_table}
WHERE month = '2026-03'
""").df()

print("Feature dataset shape:", feature_data.shape)

print("\nColumns used:")
print(feature_data.columns.tolist())

display(feature_data.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature dataset shape: (9841378, 25)

Columns used:
['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_users,ga4_engaged_sessions,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


## 2. Feature notes (meaning, missing, categorical, available-when?)

A future outcome label is created using March 2026 performance data.  
For each client-content pair, the March clicks and impressions are aggregated.

The target `future_click_label` is set to 1 when the content receives at least one click during March, and 0 otherwise. This label represents a future outcome rather than a feature available before the prediction period.

In [6]:
# ML-05 — Section 2: Create the future outcome label

# Aggregate March 2026 performance at client-content level.
future_outcome = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_clicks) AS march_clicks,
    SUM(gsc_impressions) AS march_impressions,

    CASE
        WHEN SUM(gsc_clicks) > 0 THEN 1
        ELSE 0
    END AS future_click_label

FROM {TABLES['fact_daily']}
WHERE month = '2026-03'

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

print("Future outcome rows:", len(future_outcome))

print("\nColumns:")
print(future_outcome.columns.tolist())

print("\nFirst 5 rows:")
display(future_outcome.head())

print("\nFuture label distribution:")
display(
    future_outcome["future_click_label"]
    .value_counts()
    .rename_axis("label")
    .reset_index(name="count")
)

Future outcome rows: 331437

Columns:
['client_hash_id', 'content_hash_id', 'march_clicks', 'march_impressions', 'future_click_label']

First 5 rows:


,client_hash_id,content_hash_id,march_clicks,march_impressions,future_click_label
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2.0,1140.0,1
1,client_73cda7b4e4f265ea,content_05597932fe4da067,0.0,57.0,0
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,0.0,149.0,0
3,client_73cda7b4e4f265ea,content_05434271b257bb68,6.0,1421.0,1
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,16.0,2770.0,1



Future label distribution:


,label,count
0,0,262600
1,1,68837


## 3. The leakage hunt

This experiment intentionally introduces the future target as a feature to demonstrate the effect of data leakage.

The `future_click_label` is copied into a feature named `LEAKED_FEATURE` and used by a Decision Tree classifier. The resulting accuracy is expected to be extremely high because the model has direct access to the target information.

This experiment serves as a controlled demonstration of why future outcome information must not be included in model features.

In [7]:
# ML-05 — Section 3: Deliberate Leakage Trap

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Aggregate March feature-window data at client-content level.
feature_agg = feature_data.groupby(
    ["client_hash_id", "content_hash_id"],
    as_index=False
).agg({
    "gsc_impressions": "sum",
    "gsc_clicks": "sum",
    "gsc_avg_position": "mean",
    "ga4_pageviews": "sum",
    "ga4_sessions": "sum",
    "ga4_users": "sum",
    "ga4_engaged_sessions": "sum",
    "ga4_total_engagement_sec": "sum",
    "sessions_organic": "sum",
    "sessions_direct": "sum",
    "sessions_referral": "sum",
    "sessions_social": "sum",
    "sessions_paid": "sum",
    "sessions_ai": "sum",
    "ai_chatgpt": "sum",
    "ai_perplexity": "sum",
    "ai_gemini": "sum",
    "ai_copilot": "sum",
    "ai_claude": "sum",
    "ai_meta": "sum",
    "ai_other": "sum",
    "scroll_events": "sum"
})

# Join the future outcome label.
model_data = feature_agg.merge(
    future_outcome[
        ["client_hash_id", "content_hash_id", "future_click_label"]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Model dataset shape:", model_data.shape)

# --------------------------------------------------
# DELIBERATE LEAKAGE
# --------------------------------------------------
# This feature directly copies the future label.
# It is intentionally added to demonstrate leakage.
model_data["LEAKED_FEATURE"] = model_data["future_click_label"]

X = model_data[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "LEAKED_FEATURE"
    ]
]

y = model_data["future_click_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

leak_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

leak_model.fit(X_train, y_train)

pred = leak_model.predict(X_test)

leak_accuracy = accuracy_score(y_test, pred)

print("\n=== LEAKAGE EXPERIMENT ===")
print(f"Accuracy WITH deliberate leakage: {leak_accuracy:.4f}")

Model dataset shape: (331437, 25)

=== LEAKAGE EXPERIMENT ===
Accuracy WITH deliberate leakage: 1.0000


## 4. What I excluded and why

The final experiment uses a proper temporal separation between features and the future outcome.

February 2026 data is used as the feature window, while March 2026 data is used to create the future label. This prevents March outcome information from being directly available to the model during prediction.

The honest experiment achieved an accuracy of 0.836488, while the deliberate leakage experiment achieved 1.000000. The difference demonstrates the importance of maintaining a clear time boundary between features and future outcomes.

In [10]:
# ML-05 — Section 4: Time-separated leakage check
# February 2026 = feature window
# March 2026    = future label window

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

daily_table = TABLES["fact_daily"]

# --------------------------------------------------
# 1. FEBRUARY = FEATURES
# --------------------------------------------------

february_features = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position

FROM {daily_table}
WHERE month = '2026-02'

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

print("February feature rows:", len(february_features))


# --------------------------------------------------
# 2. MARCH = FUTURE LABEL
# --------------------------------------------------

march_labels = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_clicks) AS march_clicks,
    SUM(gsc_impressions) AS march_impressions,

    CASE
        WHEN SUM(gsc_clicks) > 0 THEN 1
        ELSE 0
    END AS future_click_label

FROM {daily_table}
WHERE month = '2026-03'

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

print("March label rows:", len(march_labels))


# --------------------------------------------------
# 3. JOIN PAST FEATURES TO FUTURE LABEL
# --------------------------------------------------

time_split_data = february_features.merge(
    march_labels[
        [
            "client_hash_id",
            "content_hash_id",
            "future_click_label"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Time-separated dataset shape:", time_split_data.shape)


# --------------------------------------------------
# 4. TRAIN / TEST SPLIT
# --------------------------------------------------

honest_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

X = time_split_data[honest_features]
y = time_split_data["future_click_label"]

# Remove rows where a feature is unavailable
valid = X.notna().all(axis=1)

X = X.loc[valid]
y = y.loc[valid]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# --------------------------------------------------
# 5. HONEST MODEL
# --------------------------------------------------

honest_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

honest_model.fit(X_train, y_train)

honest_pred = honest_model.predict(X_test)

honest_accuracy = accuracy_score(
    y_test,
    honest_pred
)


# --------------------------------------------------
# 6. RESULTS
# --------------------------------------------------

comparison = pd.DataFrame({
    "experiment": [
        "Same-period experiment with leakage risk",
        "Time-separated honest experiment"
    ],
    "accuracy": [
        leak_accuracy,
        honest_accuracy
    ]
})

print("\n=== FINAL LEAKAGE AUDIT ===")
display(comparison)

print("\nTime separation:")
print("Feature window : February 2026")
print("Label window   : March 2026")

print("\nConclusion:")
print(
    "The deliberate leakage experiment achieved 1.0000 accuracy because "
    "the leaked feature directly contained the target."
)

print(
    "For the honest experiment, features are taken from February 2026 "
    "while the outcome label is taken from March 2026."
)

print(
    "This time separation prevents the model from using March outcome "
    "information when making the prediction."
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

February feature rows: 321546


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March label rows: 331437
Time-separated dataset shape: (303572, 6)

=== FINAL LEAKAGE AUDIT ===


,experiment,accuracy
0,Same-period experiment with leakage risk,1.000000
1,Time-separated honest experiment,0.836488



Time separation:
Feature window : February 2026
Label window   : March 2026

Conclusion:
The deliberate leakage experiment achieved 1.0000 accuracy because the leaked feature directly contained the target.
For the honest experiment, features are taken from February 2026 while the outcome label is taken from March 2026.
This time separation prevents the model from using March outcome information when making the prediction.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.